# Qiskit Fall Fest: Advanced Quantum Workshop 👷!
### Shor's Algorithm and Quantum Phase Estimation: The workhorse of advantage.


In [ ]:
# Install your packages
!pip install qiskit qiskit-aer pylatexenc

In [ ]:
# Import packages
import qiskit as qk
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer import AerSimulator
from qiskit.circuit.library import RZGate
import numpy as np
import matplotlib.pyplot as plt
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import math

### Background and Motivation
Computers are everywhere and sometimes, especially with the rise of Artificial Inteligence (AI), it seems like they can do anything. There are some problems that our normal (classical) computers will not be helpful in solving; not because they couldn't solve them, rather because it would take far too long (potentially tens of thousands of years) to solve. An ideal quantum computer could solve some of these problems in minutes. We study Quantum Phase Estimation today as it is a key algorithm for realizing this speedup.

Quantum Computing's (QC)'s advantage is often described by QC's exponential growth of states; where 40 classical bits has just 40 pieces of information, 40 qubits has $2^{40}$, over a trillion, distinct states with each having an associated amplitude. This explanation misses a critical consideration: restrictions, QC's operations are restricted as they must be unitary. Restrictions on operations, such as removing entanglement or using non-universal gatesets can even allow certain quantum circuits to be efficiently simulatable on classical computers. Instead of considering the statesize it is both more accurate and useful to consider speedup, how many fewer operations would be required to solve a problem.

QC algorithm's have demonstrated exponential speedup in various algorithms, Quantum Phase Estimation (QPE) is the key example. QPE is the core of Shor's algorithm, which can factor large numbers; this essentially cracks the main encroption schemes we have used for the past 50 years. QPE is also a key to quantum chemistry's interest in QC. QPE will be used in drug discovery, battery technology research, and many other critical research fields.

I hope I have convinced you to be excited about QPE, so now lets outline the questions this workshop will answer:

1.   What is QPE?
2.   How does QPE work: where does exponential speedup come from?
3.   How does Shor's algorithm use QPE?
4.   How can we implement QPE and Shor's algorithm on a quantum device?

#### 1. What is QPE?

QPE can be treated like a function with input U and $\lvert \psi \rangle$, it is assumed that $\lvert \psi \rangle$ is an eigenstate of $U$ such that $U\lvert \psi \rangle = e^{i\phi}\lvert \psi \rangle$. Its output is set on n ancilliary qubits (during this tutorial this output will be interpreted for you in histograms from $-\pi$ to $\pi$).

Instead of a long drawn out explanation, we are going to just use it and see what happens.

In [ ]:
# @title
from fractions import Fraction
# Run this cell to be able to use QPE in the next one.


def validate_and_reformat_QPE_inputs(U, psi):
    if isinstance(U, qk.circuit.Gate):
        U_num_qubits = U.num_qubits
        U_len = 2**U_num_qubits
    else:
        U_len = U.shape[0]
        U = qk.circuit.library.UnitaryGate(U)

    if len(psi) != U_len:
            raise ValueError("Psi and U must be of the same size")
    num_qubits = np.log2(len(psi))
    if num_qubits != int(num_qubits):
        raise ValueError("Psi must be a vector of length 2^n")
    return int(num_qubits), U

def QPE(U, psi, num_eval_qubits=4):
    num_qubits, U = validate_and_reformat_QPE_inputs(U, psi)

    total_qubits = num_eval_qubits+num_qubits

    circ = qk.circuit.library.phase_estimation(num_eval_qubits, U)

    qc = qk.QuantumCircuit(num_qubits)
    qc.initialize(psi, range(num_qubits))


    circ = circ.compose(qc, range(num_eval_qubits, total_qubits), front=True)
    cr = qk.ClassicalRegister(num_eval_qubits, name="my_cbits")
    circ.add_register(cr)
    circ.measure(range(num_eval_qubits), range(num_eval_qubits))

    sampler = AerSampler()
    pm = generate_preset_pass_manager(backend=AerSimulator(), optimization_level=1)
    isa_circuit = pm.run(circ)

    job = sampler.run([isa_circuit])
    result = job.result()
    counts = result[0].data.my_cbits.get_counts()
    plot_counts(counts, num_eval_qubits, label_by_pi=True)

def plot_counts(counts, num_eval_qubits, label_by_pi=False, clear_plot=True):
    if clear_plot:
        plt.close()

    if num_eval_qubits > 5:
        plt.figure(figsize=(13, 4))

    num_bins = 2**num_eval_qubits
    bin_counts = [0]*num_bins

    for bin_string in counts.keys():
        val = int(bin_string[::-1], 2)/(2**num_eval_qubits)
        bin_counts[int(np.floor_divide(val, (1/num_bins)))] += counts[bin_string]

    new_counts = {}
    if label_by_pi:
        for i in range(num_bins):
            if i < num_bins/2:
                new_counts[f"{Fraction(i, int(num_bins/2))}⋅π"] = bin_counts[i]
            else:
                new_counts[f"-{Fraction(abs(i-num_bins), int(num_bins/2))}⋅π"] = bin_counts[i]

        labels = list(new_counts.keys())[int(num_bins/2):] + list(new_counts.keys())[:int(num_bins/2)]
        values = list(new_counts.values())[int(num_bins/2):] + list(new_counts.values())[:int(num_bins/2)]
        plt.xlabel(r"Phase $(\phi),\quad \phi \in [-\pi, \pi)$")

    else:
        for i in range(num_bins):
            new_counts[f"{(i/num_bins):.3f}"] = bin_counts[i]

        labels = list(new_counts.keys())
        values = list(new_counts.values())
        plt.xlabel(r"Phase $(2\pi\phi),\quad \phi \in [0, 1)$")

    plt.bar(labels, values, edgecolor="black")

    if num_eval_qubits > 6:
        N = 2**num_eval_qubits
        accepted_label_indices = [i*N/8 for i in range(8)] + [N-1]
        custom_labels = [label if i in accepted_label_indices else "" for i, label in enumerate(labels)]
        plt.xticks(ticks=range(len(labels)), labels=custom_labels)

    plt.xticks(rotation=78)
    plt.ylabel("Counts")

    plt.show()

In [ ]:
# Run this cell after the previous cell to test QPE
Z = np.array([[1, 0], [0, -1]])
psi = np.array([1, 0])


# Try replacing psi with some of these
# psi = np.array([0, 1])
# psi = np.array([1, 1])/np.sqrt(2)
# psi = np.array([1, -1])/np.sqrt(2)
QPE(Z, psi)

In [ ]:
# Once you try those see what happens here:
RZ = RZGate(2*np.pi/5)
psi = np.array([0, 1])


# What is different, why?
# Hint: try using named parameter num_eval_qubits=5 to see what changes
QPE(RZ, psi)


<details>
<summary>Ok, fine here's a short explanation:</summary>

The point of QPE is to find $\phi$ from $U\lvert \psi \rangle = e^{i\phi}\lvert \psi \rangle$. Assuming no external error, or even bounded error, QPE guarauntees an accuracy proportional to the number of evalution qubits (note what happens when you change the number of qubits). The success of the algorithm also relies on $\lvert \psi \rangle$ being an eigenstate, you can see this through the splitting that occurs if you use $\lvert \psi \rangle = (\lvert 0 \rangle +\lvert 1 \rangle)/\sqrt{2}$. Immagine what would happen if you had an expenential number of states, the histogram would flatten.

</details>

#### 2. How does QPE work: where does exponential speedup come from?

QPE uses Quantum Fourrier Transform to change the phase into a qubit string selection. This is really just the inverse of a Discrete Fourier Transform (DFT). Fourier Transforms turn probability distrobutions into distributions of waves, ie phases. DFT does this on discrete data with discrete outputs. QFT does DFT exponentionally faster than the fastest known classical method of performing FFT.

Lets look at some examples of DFT

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt

# 1. Generate a sample signal (Sum of 50 Hz and 120 Hz sine waves)
sampling_rate = 1000  # Hz
duration = 1.0        # seconds
N = int(sampling_rate * duration)  # Total number of samples

t = np.linspace(0, duration, N, endpoint=False)



def DFT(signal):
    fft_output = np.fft.fft(signal)
    frequencies = np.fft.fftfreq(N, 1 / sampling_rate)
    amplitudes = 2.0 * np.imag(fft_output) / N

    # Plot the Frequency Spectrum
    plt.figure(figsize=(8, 4))
    plt.stem(frequencies, amplitudes, markerfmt=" ", basefmt="b")
    plt.title("Frequency Spectrum")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Real(Amplitude)")
    plt.grid()
    plt.show()

In [ ]:
# DFT on input 1
signal = np.zeros(len(t))
signal[1] = 1
DFT(signal)

In [ ]:
# DFT on single input 3
signal = np.zeros(len(t))
signal[3] = 1
DFT(signal)

In [ ]:
# DFT on sum of input 1 and input 3
signal = np.zeros(len(t))
signal[1] = 1
signal[3] = 0.5
DFT(signal)

In [ ]:
# DFT on a sine wave input
signal = np.sin(2 * np.pi * 200 * t)
DFT(signal)

In [ ]:
# DFT on a sum of sine wave inputs
signal = 3.0 * np.sin(2 * np.pi * 50 * t) + 1.5 * np.sin(2 * np.pi * 120 * t)
DFT(signal)

### 2.1: Lets build a QFT circuit:

To do this lets first define an equivalent to QFT, $QFT'$ on a computational state $\lvert x \rangle$ such as $\lvert 0 \rangle$, $\lvert 4 \rangle$, or even $\lvert 100 \rangle$:

$QFT'\lvert x \rangle = \bigotimes\limits_{k=0}^{n-1}(\lvert 0 \rangle + \exp(\frac{2\pi i2^{k}}{2^n}x)\lvert 1\rangle))$

To see what $\bigotimes\limits_{k=0}^{n-1}$ does, lets expand the series:

$QFT'\lvert x \rangle = (\lvert 0 \rangle + \exp(\frac{2\pi i2^{0}}{2^n}x)\lvert 1\rangle) \otimes (\lvert 0 \rangle + \exp(\frac{2\pi i2^{(n-2)}}{2^n}x)\lvert 1\rangle) \otimes ... \otimes (\lvert 0 \rangle + \exp(\frac{2\pi i2^{0}}{2^n}x)\lvert 1\rangle)$

Hopefully you can see that we can break this down into indivdual operations to change the phase of each qubit:
$QFT'\lvert x \rangle= ∏\limits_{k=0}^{n-1}U_k\lvert x \rangle=U_0U_1...U_{n-1}\lvert x \rangle$

$U_k\lvert x \rangle = \lvert x_{n-1} ... x_{k+1} \rangle \otimes (\lvert 0 \rangle + \exp(\frac{2\pi i2^{k}}{2^n}x)\lvert 1\rangle)) \otimes \lvert x_{k-1} ... x_0 \rangle$


We need to implement these $U_k$.


**Example:** 3 qubit circuit. First start with $U_2$, trust me its the easiest.

$U_2\lvert x \rangle = \lvert x_{2} x_{1} \rangle \otimes (\lvert 0 \rangle + \exp(\frac{2\pi i2^{2}}{2^3}x)\lvert 1\rangle))= \lvert x_{2} x_{1} \rangle \otimes (\lvert 0 \rangle + \exp(x\pi i)\lvert 1\rangle))$

Here we see if x is even the exponential is $+1$, if x is odd $-1$, x being even or odd is controlled by the least significant bit being $\lvert 0 \rangle$ or $\lvert 1 \rangle$. This is just a Hadamard gate!


$U_1\lvert x \rangle = \lvert x_{2} x_{1} \rangle \otimes (\lvert 0 \rangle + \exp(\frac{x\pi i}{2})\lvert 1\rangle))$

Now lets look at the binary of x: $x=x_2*2^2+x_1*2^1+x_0*2^0$ if $x_2$ = 1 we add $2^2$, which will be equivalent to adding $2\pi$ to our phase, which can be ignored, in general we can safely ignore the contributions of bits larger than (n-1)-k, as they will all have this same effect. Take $x_1$ it will add $\pi$ phase similarly to what we saw for $U_2$, this again can be accomplished with a Hadamard gate on said wire. Next, we need to account for the phase caused by the 0th wire, which could be done using a controlled RZ operation, which contributes $\frac{\pi}{2}$ phase.

You might note, however, that there would be an undesirable phase added to the $\lvert 0 \rangle$ state of the target qubit; this is commonly ignored, but we will use a different gate to fix this. We will use the controlled phase gate, which only applies phase to $ \lvert11\rangle$. Can you think of how you could recreate this with CRZ and single qubit gates?

$U_0\lvert x \rangle = \lvert x_{2} x_{1} \rangle \otimes (\lvert 0 \rangle + \exp(\frac{x\pi i}{4})\lvert 1\rangle))$

This has a similar form, but requires a total of 3 gates. The first two have been implemented, try implementing $U_0$ bellow



In [ ]:
# Here is a circuit with U1 and U2
circ = qk.QuantumCircuit(2)

circ.h(1)
circ.cp(np.pi / 2, 0, 1)
circ.h(0)



# Use this circuit object to make U0
U0_circ = qk.QuantumCircuit(3)

# TODO: Fill in answer here:


# This merges the two circuits
circ = U0_circ.compose(circ, [0,1])

# This will draw the circuit, run the cell to see your circuit!
circ.draw("mpl")

To do this generally, we must make a $U_{(n-1)-k}$ circuit construction function then a general circuit composition function.

Fill in code in the following cell to build these functions.

In [ ]:
def subcircuit_constructor(n, k):
    """
    This function will take in k and n and will return a circuit object that implements U_{(n-1)-k}
    """
    num_qubits = n-k
    if num_qubits<=0:
        raise AttributeError("n must be greater than k, as n-1 is the maximum k")

    subcirc = qk.QuantumCircuit(num_qubits)
    subcirc.h(0) # TODO: Replace with correct value

    # loop through qubits bottom to top (order doesn't matter)
    for working_qubit in range(num_qubits-2, -1, -1):
        # TODO: Modify values
        phase_change = np.pi / 2 # Replace with correct value
        ctrl_bit = 0 # Replace with correct value
        target_bit = 1 # Replace with correct value

        subcirc.cp(phase_change, ctrl_bit, target_bit)
    return subcirc

subcircuit_constructor(3,0).draw("mpl")

In [ ]:
def get_QFT_prime_circuit(n):
    current_circuit = qk.QuantumCircuit(n)
    for k in range(n):
        k_subcircuit = subcircuit_constructor(n, k)

        # TODO: Modify values
        k_circuit_qubits = [] # Replace with correct qubit selections
        is_front = False # Choose if True or False

        current_circuit =current_circuit.compose(k_subcircuit, k_circuit_qubits, front=is_front)
    return current_circuit

get_QFT_prime_circuit(3).draw("mpl")


At the start we said this was $QFT'$, not QFT, the difference is that this flips the  $QFT'$ actually flips the bit ordering, so now to get the required output for QFT, this can be done with swap gates. Fix the following function to create your final QFT circuit.

In [ ]:
def get_QFT_circuit(n):
    current_circuit = get_QFT_prime_circuit(n)
    for i in range(n//2):
        pass # TODO: Add SWAP gates

    return current_circuit

get_QFT_circuit(3).draw("mpl")

In [ ]:
# Always good to check your work, you can do so here:
def check_QFT_constructor_correct(n):
    my_qft_op = qk.quantum_info.Operator(get_QFT_circuit(n))
    qiskit_qft_op = qk.quantum_info.Operator(qk.circuit.library.QFTGate(n))

    print(f"QFT Correct for n={n}: {my_qft_op.equiv(qiskit_qft_op)}")

check_QFT_constructor_correct(8)
check_QFT_constructor_correct(9)

### 2.2 Going Back to QPE from QFT

QPE uses QFT to turn the phase $\phi$ from $U\lvert \psi \rangle = e^{i\phi}\lvert \psi \rangle$ from phase to a sampleable discrete amplitude distribution. You may note that we are going from phase to sampling amplitude, which is the oposite of our QFT circuit. We can simply reverse the circuit to get its adjoint, which in the case of unitaries such as QFT is the inverse. Therebye applying $QFT^†$, which takes a phase distribution and turns it into a sampling probability distribution. We must then set up a set of qubits to encode a phase distribution representing $\phi$.

An ideal phase encoding may look like: $(\lvert 0 \rangle + e^{2\pi i\phi 2^{(n-1)}} \lvert 1 \rangle) \otimes (\lvert 0 \rangle + e^{2\pi i\phi 2^{(n-2)}} \lvert 1 \rangle) \otimes ... \otimes (\lvert 0 \rangle + e^{2\pi i\phi 2} \lvert 1 \rangle) \otimes (\lvert 0 \rangle + e^{2\pi i\phi} \lvert 1 \rangle)$.

But how can we get that? Lets look what happens wen we have a controlled operation on $U\lvert \psi \rangle$:

$$c-U\lvert 0 \rangle\lvert \psi \rangle = \lvert 0 \rangle \otimes I\lvert \psi \rangle= \lvert 0 \rangle\lvert \psi \rangle, \quad c-U\lvert 1 \rangle\lvert \psi \rangle = \lvert 1 \rangle \otimes U\lvert \psi \rangle= e^{i\phi}\lvert 0 \rangle \lvert \psi \rangle$$

Also it is pretty obvious that $U^x \lvert \psi \rangle = e^{ix\phi} \lvert \psi \rangle$
Therefore we can use controlled exponential unitaries to implement the correct phase information on each qubit. For now lets just use a naive approach where we get $c-U^x$, by repeating x times. Obviously, this will grow exponentially, but it will help us build simple QPE circuits for now.

In [ ]:
def naive_QPE(num_eval_qubits, U, psi,  initiate_state=True, measure_output=True):
    num_qubits, U = validate_and_reformat_QPE_inputs(U, psi)

    circ = qk.QuantumCircuit(num_eval_qubits + num_qubits)

    # initiate state psi
    if initiate_state:
        circ.initialize(psi, range(num_eval_qubits, num_eval_qubits + num_qubits))

    for eval_qubit in range(num_eval_qubits):
        # TODO: There is a gate we need to add to turn the intial state |0> into a more useful state.
        # You can add this gate here.


        for i in range(2**(eval_qubit)):
            cU = U.control()

            # TODO: Modify values
            control_qubit = 0 # Replace with correct control qubit
            target_qubits = [] # Replace with correct target qubits
            circ.append(cU, [control_qubit]+target_qubits)

    # Now we add back QFT circuit
    my_qft_circ = get_QFT_circuit(num_eval_qubits) # Remember to take its adjoint
    # TODO: add it back to the circuit here



    # Oh no our inversed QFT flipped the outputs again, we can fix this with another set of swap gates.
    # TODO: add code for SWAP gates here:



    # Measurement stage:
    if measure_output:
        circ.barrier(label="Measure")
        cr = qk.ClassicalRegister(num_eval_qubits, name="my_cbits")
        circ.add_register(cr)
        circ.measure(range(num_eval_qubits), range(num_eval_qubits))
    return circ

RZ = RZGate(2*np.pi/5)
psi = np.array([0, 1])

naive_QPE(3, RZ, psi).draw("mpl")

In [ ]:
# Again, lets check against qiskits implementation here:
def check_QPE_constructor_correct(num_eval_qubits, U, psi):
    my_qpe_circ = naive_QPE(num_eval_qubits, U, psi,  initiate_state=False, measure_output=False)
    qiskit_qpe_circ = qk.circuit.library.phase_estimation(num_eval_qubits, U)

    my_qpe_op = qk.quantum_info.Operator(my_qpe_circ)
    qiskit_qpe_op = qk.quantum_info.Operator(qiskit_qpe_circ)

    print(f"QPE Correct: {my_qpe_op.equiv(qiskit_qpe_op)}")

RZ = RZGate(2*np.pi/5)
psi = np.array([0, 1])

check_QPE_constructor_correct(5, RZ, psi)

Getting Back on track: we can see now that if you give me a black box unitary and you tell me to run QPE, we get an exponential run time. Thankfully, useful algorithms figure out how to efficiently implement their controlled unitaries and controlled exponential unitaries efficiently, such as Shors algorithm.

## 3. Shor's Algorithm:

Shor's algorithm takes an input composite (non-prime) integer and outputs its factors. The input integer is stored in binary and classical solvers to this problem grow exponentially with the number of bits used to store the number. Modern encryption has relied on keys between 256 bits to 4096 bits, with RSA-2048 being the current standard. Here we are going to factor 15, which is a 4 bit number.

Shor's algorithm is a bit complex, but for now we just need to know our quantum computer is used to solve the order finding problem. The order finding problem takes inputs $N$ (the integer we are attempting to factor) and $a$ (a random integer, less than N and greater than 1); it then outputs $r$ where $a^r=1\mod N$. How does this relate to our inputs $\lvert \psi \rangle$, $U$ and output $\phi$ for QPE?

Multiplication is actually a pretty common operation in QC so lets take $U=M_a$ where $M_a\lvert x \rangle = \lvert (a\cdot x) \mod N \rangle|\quad \forall x\lt N$. As long as $M_a$ seperates $x\geq N$ and $x\lt N$ $M_a$ is pretty easy to see  it is unitary and we can ignore the behavour of larger x.  Additionally $(M_a)^{2^k}\lvert x \rangle =\lvert [(a^{2^k})\cdot x] \mod N \rangle$ or $M_{b}, \quad b = a^{2^k}$, yay we can exploit the structure to not have exponential runtime!


We just need to implement our $M_a$ operations, in general this can be done by creating a circuit for $a*2^j, \quad \forall j \in \{0,\dots,n-1\}$ where $n$ is the number of qubits, this can be done with CNOT and SWAP (which can be decomposed into CNOTs) gates, and making it controlled by the QFT bits can be done by making these double controlled.


### 3.1, implementing Shor's $U$
Lets implement $M_2$ for N=15 for now:

In [ ]:
def get_basic_M_2():
    M2 = qk.QuantumCircuit(4)
    # Order may first appear stange, but there is a reason for the ordering

    # 2^2 = 4 = ob0010|  4*2 = 8 = 0b0001
    # TODO: Implement gates to apply this (hint: think swap)



    # 2^1 = 2 = ob0100|  2*2 = 4 = 0b0010
    # Note: Selection of this ordering does not break previous input
    # TODO: Implement gates to apply this



    # 2^0 = 1 = 0b1000 |  1*2  = 2 = 0b0100
    # TODO: Implement gates to apply this



    # 2^3 = 8 = 0b0001 |  8*2 % 15  = 1 = 0b1000
    # Look at the gates you have already added, this came out naturally.
    return M2
M2 = get_basic_M_2()
M2.draw("mpl")


In [ ]:
# Now expand the SWAP gates and add an extra control
# CNOT[0,1] CNOT[1,0], CNOT[0,1] = SWAP[0,1]

def get_controled_M_2():
    cM2 = qk.QuantumCircuit(5)

    # cM2.swap(2, 3) expanded
    cM2.ccx(0, 3, 4)
    cM2.ccx(0, 4, 3)
    cM2.ccx(0, 3, 4)
    #Note, +1 to indexes as first wire is now control wire

    # cM2.swap(1, 2) expanded
    # Now expand the 1,2 SWAP.
    # TODO: expand cM2.swap(1, 2) and add second control.



    # Now expand the 0,1 SWAP.
    # TODO: expand cM2.swap(0, 1) and add second control.



    return cM2



# I wrote this one for you to save time
def get_controled_M_4():
    cM4 = qk.QuantumCircuit(5)

    # 2,8 now swap
    cM4.ccx(0, 2, 4)
    cM4.ccx(0, 4, 2)
    cM4.ccx(0, 2, 4)

    # 1,4 now swap
    cM4.ccx(0, 1, 3)
    cM4.ccx(0, 3, 1)
    cM4.ccx(0, 1, 3)

    return cM4


### 3.2, implementing Shor's algorithm

Now we are almost ready to implement Shors algorithm. We just need to select $\lvert \psi \rangle$. $M_a$ has eigenvector/eigenvalue pairs we can exploit:
$$j \in \{0, \dots, r-1\}: \quad \lvert \psi_j\rangle = \frac{1}{\sqrt{r}} \sum\limits_{k=0}^{r-1 }\exp\left(2\pi i \frac{j}{r}\cdot k\right)\lvert a^k\rangle$$

It would be ideal to take $\lvert\psi\rangle = \lvert\psi_1\rangle$, estimating phase $\theta=1/r$ would give us our answer immediately, but unfortunately there is no free lunch, (well there was, but not here). Instead lets just try the state $\lvert 1\rangle$. This may seem naive, but let's express $\lvert 1\rangle$ in $M_a$'s eigenbasis:
$$\lvert 1\rangle = \frac{1}{\sqrt{r}}\sum\limits_{j=0}^{r-1}\left[\frac{1}{\sqrt{r}} \sum\limits_{k=0}^{r-1 }\exp\left(2\pi i \frac{j}{r}\cdot k\right)\lvert a^k\rangle\right]\,.$$

Thats right, everything, but $\lvert a^0 \rangle$ cancels when summed over. This will be ideal for now.

In [ ]:
def Shors_alg_N_15_a_2(num_eval_qubits = 4, measure_output=True):
    num_qubits = 4

    cM2 = get_controled_M_2()
    cM4 = get_controled_M_4()
    # What about cM16, 16=2^(2^3)?

    circ = qk.QuantumCircuit(num_eval_qubits + num_qubits)

    # initiate state psi
    circ.x(num_eval_qubits)


    for eval_qubit in range(num_eval_qubits):
        # TODO: There is a gate we need to add to turn the intial state |0> into a more useful state.
        # You can add this gate here.
        pass

    # TODO: Modify values
    # Add back your controls, there are only 2, why?
    circ.compose(cM2, [], front=None)
    circ.compose(cM4, [], front=None)



    circ.barrier()
    my_qft_circ = get_QFT_circuit(num_eval_qubits) # Remember to take its adjoint
    # TODO: Add in your QFT circuit to current circuit



    # Again, our inversed QFT flipped the outputs again, we can fix this with another set of swap gates.
    # TODO: Add code for SWAP gates here:


    # Measurement stage:
    if measure_output:
        circ.barrier(label="Measure")
        cr = qk.ClassicalRegister(num_eval_qubits, name="my_cbits")
        circ.add_register(cr)
        circ.measure(range(num_eval_qubits), range(num_eval_qubits))
    return circ

Shors_alg_N_15_a_2().draw("mpl")

# 3.3, running Shor's Phase Estimation

Due to our selection of $\lvert \psi \rangle = \lvert 1 \rangle$ running QFT we expect an even distribution of sampling $r$ values separated by $k/r$ where $k \in \{0, \dots, r-1\}$. This will obviously allow us to learn $r$ with a high degree of confidence after a reasonable number of runs (Note: not only the number of spikes, but the spacing of spikes contributes to the expected probability distribution)

In [ ]:
num_eval_qubits = 5
circ = Shors_alg_N_15_a_2(num_eval_qubits)


sampler = AerSampler()
pm = generate_preset_pass_manager(backend=AerSimulator(), optimization_level=1)
isa_circuit = pm.run(circ)

job = sampler.run([isa_circuit])
result = job.result()
counts = result[0].data.my_cbits.get_counts()
plot_counts(counts, num_eval_qubits)

In [ ]:
def get_g_from_r(r):
    N = 15
    common_divisor_check_val = int(2**(r/2))+1

    g = math.gcd(N, common_divisor_check_val)

    print(f"p={g}, q={int(N/g)} => p*q=15")

# What do you think r is?
get_g_from_r(r = None) # TODO: fill in r

<details>
<summary>A short explanation of Shor's Algorithm:</summary>

Shor's algorithm reduces the factoring problem to the order-finding problem, in a way Shor's algorithm really solves the order finiding problem. What is the order-finding problem? Thank you for asking.

The order finding problem: take two positive integer inputs $a$ and $N$ and find the smallest positive integer $r$ where $a^r=1\mod N$.

Shor's algorithm takes positive integer input $N$, and tries to find two integer factors $p$ and $q$ larger than 1. It is assumed $N$ is non-even as that has a trivial answer and that $N$ is not the power of a prime number, as there is an efficient classical algorithm to check this. The algorithm selects a random integer $2\leq a \lt N$; a classical check for the greatest common devisor (gcd) of $N$ and $a$ is commonly performed here ($g=gcd(N,a)$ if $g>1$ then $g=p$) but this is not necessary. Now we run an a quantum algorithm to find $r$. We then check if r is even then check $g=gcd(N,a^{r/2}+1)$  and if $g>1$ then $g=p$. This process is repeated until a valid non-trivial $g$ is found. The algorithm has an over 50% chance of success on any given $a$ selection.
</details>

## 4. Running on real hardware

You will need to sign up for free compute time here: https://quantum.cloud.ibm.com/registration.

Here you can take the code you wrote and run it on real hardware!

In [ ]:
# Install runtime to run on real hardware
!pip install qiskit-ibm-runtime

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.executor_sampler import Sampler

In [ ]:
your_api_key = "deleteThisAndPasteYourAPIKeyHere"
your_crn = "deleteThisAndPasteYourCRNHere"

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=your_api_key,
    instance=your_crn,
    name="fallfest-2026",
)

In [ ]:
service = QiskitRuntimeService(name="fallfest-2026")
backend = service.least_busy(operational=True, simulator=False)

# Check that the account has been saved properly
print(service.saved_accounts())

In [ ]:
num_eval_qubits = 4
circ = Shors_alg_N_15_a_2(num_eval_qubits)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circ)
sampler = Sampler(mode=backend)

job = sampler.run([isa_circuit])
result = job.result()
counts = result[0].data.my_cbits.get_counts()
plot_counts(counts, num_eval_qubits)

This workshop was based on Tutorials from IBM's Quantum Platform including

[1] https://quantum.cloud.ibm.com/learning/en/modules/computer-science/qft

[2] https://quantum.cloud.ibm.com/learning/en/courses/utility-scale-quantum-computing/quantum-phase-estimation

[3] https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm